# Data manipulation demo

Anton Antonov  
September 2026

---

## Introduction

This notebook has data manipulation examples using the Raku package "H2O::Client".

---

## Setup

In [14]:
use H2O::Client;

use Data::ExampleDatasets;
use Data::Generators;
use Data::Reshapers;
use Data::Summarizers;
use Data::TypeSystem;

use Statistics::Distributions;

use JavaScript::D3;

In [15]:
#% javascript
require.config({
  paths: {
    d3: "https://d3js.org/d3.v7.min",
    d3_3d: "https://unpkg.com/d3-3d@2.0.2/build/d3-3d"
  }
});

require(["d3", "d3_3d"], function(d3, d3_3d) {
  window.d3 = d3;
  window.d33d = d3_3d || window.d33d || {};
  console.log("d3:", d3.version, "d3_3d:", Object.keys(window.d33d));
});

In [16]:
#% js
js-d3-list-line-plot(100.rand xx 30, background => 'none')

----

## H2O client

It is assumed that a running H2O cluster is accessible via "http://localhost:54321":

In [17]:
my $h2o = H2O::Client('http://localhost:54321')

H2O::Client.new(base-url => "http://localhost:54321", timezone => -14400, transport => H2O::Client::Transport.new(base-url => "http://localhost:54321", timeout => 10))

----

## Upload Raku session data

Random tabular dataset created with `random-tabular-dataset` of "Data::Generators":

In [18]:
my @data = random-tabular-dataset(300, <activity name date value>, 
    generators => {
        activity => <walk sleep eat attack run-away ambush>,
        name => do with (random-pet-name(7) Z=> [5, 5, 3, 3, 2, 2, 1]).Mix { -> $size { $_.roll($size).Array } },
        date => { random-date-time(size => $_)},
        value => { random-variate(NormalDistribution.new(4, 1), $_ ) }
    });

deduce-type(@data)

Vector(Struct([activity, date, name, value], [Str, DateTime, Str, Num]), 300)

Import to H2O's cluster:

In [19]:
$h2o.data-import(@data, destination-frame => 'pets')

H2O::Job<$0301c0a801a732d4ffffffff$_bb0db58d5335e84378685d02e971429a>[RUNNING 0%]

See "pets" it in H2O's frames:

In [20]:
$h2o.frames

[H2O::Frame<pets>[300 × 4]]

Get a proxy reference:

In [21]:
my $pets = $h2o.frame('pets')

H2O::Frame<pets>[300 × 4]

Show the column names and types of "pets":

In [22]:
.say for $pets.types.sort(*.key)

activity => enum
date => string
name => enum
value => real


----

## Import file from the Web

In [35]:
my $url = 'https://s3.amazonaws.com/h2o-public-test-data/smalldata/airlines/allyears2k_headers.zip';

https://s3.amazonaws.com/h2o-public-test-data/smalldata/airlines/allyears2k_headers.zip

Import the CSV into an H2O data frame. The source column named `class` is renamed to `species` while parsing:

In [39]:
my $airlines = $h2o.import-file(
    $url,
    :job,
    destination-frame => 'airlines.hex',
).wait.result;

H2O::Frame<airlines.hex>[43978 × 31]

Get the dimensions and summary of the imported frame:

In [40]:
$airlines.dimensions(:pairs);

{columns => 31, rows => 43978}

Group the rows by origin and destination the number of rows in each group:

In [73]:
#%html
my $counts = $airlines.expression
    .group-by(<Origin Dest>)
    .count
    .materialize("airlines-by-origin-dest{(^100_000).pick}.hex");

#%html
$counts.Array
==> { .sort(- *<nrow>).head(16) }()
==> to-html(field-names => <Origin Dest nrow> )    

Origin,Dest,nrow
DEN,PHX,1961
BUR,PHX,1376
AUS,PHX,782
ABQ,PHX,742
ATL,PHX,600
DFW,PHX,577
CMH,PHX,565
CVG,PHX,498
COS,PHX,446
STL,PHL,394


Make a Pareto principle plot:

In [92]:
#%js
$counts.Array.map(*<nrow>)
==> pareto-principle-statistic()
==> js-d3-list-line-plot(
    background => 'none', 
    title => 'Pareto principle for Origin-Destination counts',
    plot-label-color => 'Gray',
    :6stroke-width,
    :grid-lines
)